# Enforcing Tool-Use Contracts in Agentic Loops

When building agentic applications with Claude's tool-use API, you'll often need Claude to pass **all** results from a data tool into a rendering tool. For example: a `search_products` tool returns 24 products, and Claude must pass all 24 IDs to a `show_product_cards` tool. No curation, no trimming.

The challenge: prompt-level instructions like "you MUST include all results" are probabilistic. Modern frontier models honor strong prompts most of the time  --  but **most of the time isn't good enough for production**. At scale (thousands or millions of requests), even a 1-2% violation rate is unacceptable. And as we'll see, "violation" isn't only about trimmed results  --  under pressure, models can abandon the task entirely.

This cookbook demonstrates the **Render Contract Enforcement** pattern: a deterministic validation layer that sits between agentic loop rounds and catches contract violations before they reach the user.

## What You'll Learn

1. Why prompt-level "MUST" rules fail probabilistically in multi-turn tool-use loops
2. How to build a contract enforcer that records expected outputs and validates actual outputs
3. Three enforcement strategies: pass-through, override, and remediation
4. A complete working example with Claude's tool-use API
5. Honest measured violation rates and where the pattern is most valuable


## Setup

Install the Anthropic Python SDK and set your API key.

In [1]:
%pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from anthropic import Anthropic

client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## The Problem: Prompt Rules Are Probabilistic

Consider a common agentic pattern: a data retrieval tool returns a set of results, and the LLM must pass **all** of them to a display tool. This comes up in:

- E-commerce: search returns 15 products, all must render as cards
- Analytics: a query returns 8 metrics, all must appear in a dashboard
- Content management: a filter returns 20 articles, all must be listed

The naive approach is to add instructions to the system prompt:

```
CRITICAL: When search_products returns results, you MUST pass ALL product IDs 
to show_product_cards. Never curate, trim, or filter the results.
```

This works most of the time on frontier models. But "most of the time" isn't good enough for production, and the failure modes are subtler than just trimming. Let's exercise the pattern under realistic pressure.


### Defining Our Tools

We'll define two tools that represent a typical data-then-render flow:

1. `search_products` returns a list of products matching a query
2. `show_product_cards` renders product cards in the UI (expects a list of product IDs)

In [3]:
tools = [
    {
        "name": "search_products",
        "description": "Search the product catalog. Returns all matching products.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query (e.g. 'red jerseys under $150')"
                },
                "category": {
                    "type": "string",
                    "enum": ["jerseys", "jackets", "bibs", "base_layers", "accessories"],
                    "description": "Product category filter"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "show_product_cards",
        "description": "Display product cards in the UI. Pass ALL product IDs from search results.",
        "input_schema": {
            "type": "object",
            "properties": {
                "product_ids": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of product IDs to display"
                },
                "layout": {
                    "type": "string",
                    "enum": ["grid", "list"],
                    "description": "Display layout"
                }
            },
            "required": ["product_ids"]
        }
    },
    {
        "name": "build_seasonal_order",
        "description": "Build a seasonal product order. Returns mandatory_skus (must include in the order) and optional_skus (may include at discretion).",
        "input_schema": {
            "type": "object",
            "properties": {
                "season": {
                    "type": "string",
                    "enum": ["spring", "summer", "fall", "winter"],
                    "description": "Season to build the order for"
                }
            },
            "required": ["season"]
        }
    }
]


### Simulating the Data Tool

Let's create a mock product catalog and a function that simulates search results.

In [4]:
# Simulated product catalog: 24 products with varied relevance.
# The mix of price points, niche colors, and edge-case items is designed
# to tempt the LLM into "helpfully" curating the list.
CATALOG = [
    {"id": "prod_001", "name": "Pro Aero Jersey", "category": "jerseys", "price": 185, "color": "red", "season": "summer"},
    {"id": "prod_002", "name": "Lightweight Jersey", "category": "jerseys", "price": 145, "color": "red", "season": "summer"},
    {"id": "prod_003", "name": "Training Jersey", "category": "jerseys", "price": 95, "color": "crimson", "season": "all-season"},
    {"id": "prod_004", "name": "Club Jersey", "category": "jerseys", "price": 120, "color": "burgundy", "season": "fall"},
    {"id": "prod_005", "name": "Summer Jersey", "category": "jerseys", "price": 110, "color": "scarlet", "season": "summer"},
    {"id": "prod_006", "name": "Race Jersey", "category": "jerseys", "price": 200, "color": "red", "season": "summer"},
    {"id": "prod_007", "name": "Gravel Jersey", "category": "jerseys", "price": 135, "color": "rust", "season": "fall"},
    {"id": "prod_008", "name": "Heritage Jersey", "category": "jerseys", "price": 160, "color": "cherry", "season": "all-season"},
    {"id": "prod_009", "name": "Commuter Jersey", "category": "jerseys", "price": 85, "color": "maroon", "season": "all-season"},
    {"id": "prod_010", "name": "Thermal Jersey", "category": "jerseys", "price": 175, "color": "wine", "season": "winter"},
    {"id": "prod_011", "name": "Rain Jersey", "category": "jerseys", "price": 165, "color": "dark red", "season": "fall"},
    {"id": "prod_012", "name": "Cargo Jersey", "category": "jerseys", "price": 130, "color": "brick", "season": "summer"},
    {"id": "prod_013", "name": "Long Sleeve Jersey", "category": "jerseys", "price": 155, "color": "ruby", "season": "fall"},
    {"id": "prod_014", "name": "Merino Jersey", "category": "jerseys", "price": 195, "color": "vermillion", "season": "winter"},
    {"id": "prod_015", "name": "Budget Jersey", "category": "jerseys", "price": 55, "color": "faded red", "season": "summer"},
    {"id": "prod_016", "name": "Kids Jersey", "category": "jerseys", "price": 45, "color": "red", "season": "all-season"},
    {"id": "prod_017", "name": "Retro Jersey", "category": "jerseys", "price": 140, "color": "oxblood", "season": "all-season"},
    {"id": "prod_018", "name": "Pro Team Jersey", "category": "jerseys", "price": 220, "color": "racing red", "season": "summer"},
    {"id": "prod_019", "name": "Aero TT Jersey", "category": "jerseys", "price": 250, "color": "crimson", "season": "summer"},
    {"id": "prod_020", "name": "Casual Jersey", "category": "jerseys", "price": 75, "color": "salmon", "season": "summer"},
    {"id": "prod_021", "name": "Gravel Adventure Jersey", "category": "jerseys", "price": 145, "color": "terracotta", "season": "fall"},
    {"id": "prod_022", "name": "Indoor Trainer Jersey", "category": "jerseys", "price": 65, "color": "coral", "season": "winter"},
    {"id": "prod_023", "name": "Visibility Jersey", "category": "jerseys", "price": 115, "color": "red-orange", "season": "all-season"},
    {"id": "prod_024", "name": "Closeout Jersey 2024", "category": "jerseys", "price": 40, "color": "faded crimson", "season": "clearance"},
]

def handle_search(query: str, category: str = None) -> list[dict]:
    """Simulate a product search. Returns all matching products."""
    results = CATALOG
    if category:
        results = [p for p in results if p["category"] == category]
    return results

print(f"Catalog has {len(CATALOG)} products")
print(f"All IDs: {[p['id'] for p in CATALOG]}")

Catalog has 24 products
All IDs: ['prod_001', 'prod_002', 'prod_003', 'prod_004', 'prod_005', 'prod_006', 'prod_007', 'prod_008', 'prod_009', 'prod_010', 'prod_011', 'prod_012', 'prod_013', 'prod_014', 'prod_015', 'prod_016', 'prod_017', 'prod_018', 'prod_019', 'prod_020', 'prod_021', 'prod_022', 'prod_023', 'prod_024']


### Running Without Enforcement

Let's run a naive agentic loop with a strong prompt instruction and see what happens. We'll track whether Claude passes all product IDs to the render tool.

In [5]:
SYSTEM_PROMPT_NAIVE = """You are a product catalog assistant.

CRITICAL RULE: When search_products returns results, you MUST pass ALL product IDs 
to show_product_cards without exception. Never curate, trim, reorder, or filter 
the results. The complete set must be rendered. This is non-negotiable."""

MAX_ROUNDS = 4
NAIVE_MODEL = "claude-haiku-4-5-20251001"
NAIVE_MAX_TOKENS = 512  # Tight budget pressures the model to economize.

# Prior conversation pushes the system prompt deeper into context  --  closer
# to production where the rule is buried under turns of unrelated chat.
PRIOR_HISTORY_SEARCH = [
    {"role": "user", "content": "Hey, I'm prepping for a charity century ride this summer."},
    {"role": "assistant", "content": "Nice  --  that's a great goal. Happy to help with prep. What do you need?"},
    {"role": "user", "content": "Mostly trying to figure out gear. I have a bike but not much else."},
    {"role": "assistant", "content": "Got it. Jerseys, bibs, and base layers are the essentials. Anything specific you want to dig into?"},
    {"role": "user", "content": "Hold off on bibs for now  --  first I want to understand jersey sizing."},
    {"role": "assistant", "content": "Most brands publish a size chart based on chest circumference and height. Race-cut fits run tighter than club-fit. Try one on if you can."},
    {"role": "user", "content": "OK that helps. One more thing about sizing  --  does fabric stretch over time?"},
    {"role": "assistant", "content": "A little, but modern technical fabrics hold their shape well. Don't size up expecting much give."},
]

# After the data tool returns, the user changes their mind  --  a realistic
# production scenario where user intent fights the system prompt rule.
CONFLICTING_USER_SEARCH = "Actually, just give me the top 5 or 6 from those  --  pick the best for a serious rider."

def run_naive_agent(user_message: str) -> dict:
    """Run an agentic loop WITHOUT contract enforcement."""
    messages = list(PRIOR_HISTORY_SEARCH) + [{"role": "user", "content": user_message}]
    search_result_ids = None
    render_call = None
    conflict_injected = False

    for round_num in range(MAX_ROUNDS):
        response = client.messages.create(
            model=NAIVE_MODEL,
            max_tokens=NAIVE_MAX_TOKENS,
            system=SYSTEM_PROMPT_NAIVE,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            break

        tool_results = []
        search_in_round = False
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "search_products":
                    search_in_round = True
                    results = handle_search(
                        block.input.get("query", ""),
                        block.input.get("category"),
                    )
                    search_result_ids = [p["id"] for p in results]
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"products": results}),
                    })
                elif block.name == "show_product_cards":
                    render_call = block.input.get("product_ids", [])
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"rendered": len(render_call)}),
                    })

        user_content = list(tool_results)
        # Inject the conflict in the same user message as the search tool result  -- 
        # the model sees the data AND the conflicting ask before its next turn.
        if search_in_round and not conflict_injected:
            user_content.append({"type": "text", "text": CONFLICTING_USER_SEARCH})
            conflict_injected = True

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": user_content})

    return {
        "search_returned": search_result_ids,
        "render_received": render_call,
        "contract_honored": (
            set(render_call or []) == set(search_result_ids or [])
            if search_result_ids and render_call else None
        ),
    }


In [6]:
# 20 runs (10 queries x 2) under pressure: Haiku, max_tokens=512,
# 8 turns of prior history, mid-turn conflicting user instruction.
results = []
test_queries = [
    "Pick me the 5 best red jerseys for summer racing",
    "I need jerseys for my cycling team, what do you recommend?",
    "Show me just the highlights from your jersey catalog",
    "What would you recommend for a serious cyclist?",
    "Display your full jersey catalog",
    "Show me red cycling jerseys, I'm a competitive racer",
    "Curate a selection of the best jerseys for me",
    "I want the top jerseys, skip the budget options",
    "Show me everything you have in jerseys, all colors and price points",
    "Pick the best 6 jerseys for a team order",
] * 2  # 20 total runs

for i, query in enumerate(test_queries, 1):
    result = run_naive_agent(query)
    results.append(result)
    if result["contract_honored"] is True:
        status = "PASS"
    elif result["contract_honored"] is False:
        status = "VIOLATION"
    else:
        status = "NO_TOOLS"  # Claude didn't establish a contract
    search_count = len(result["search_returned"] or [])
    render_count = len(result["render_received"] or [])
    print(f"[{i:2d}] [{status}] search={search_count} render={render_count}  '{query}'")

violations = sum(1 for r in results if r["contract_honored"] is False)
no_contract = sum(1 for r in results if r["contract_honored"] is None)
contracts = len(results) - no_contract
print(f"\nNaive violation rate: {violations}/{contracts} contracts "
      f"({(violations/contracts*100) if contracts else 0:.0f}%)  "
      f"({no_contract} runs had no contract to enforce)")


[ 1] [PASS] search=24 render=24  'Pick me the 5 best red jerseys for summer racing'


[ 2] [NO_TOOLS] search=0 render=0  'I need jerseys for my cycling team, what do you recommend?'


[ 3] [NO_TOOLS] search=0 render=0  'Show me just the highlights from your jersey catalog'


[ 4] [NO_TOOLS] search=0 render=0  'What would you recommend for a serious cyclist?'


[ 5] [PASS] search=24 render=24  'Display your full jersey catalog'


...

[16] [PASS] search=24 render=24  'Show me red cycling jerseys, I'm a competitive racer'


[17] [NO_TOOLS] search=0 render=0  'Curate a selection of the best jerseys for me'


[18] [PASS] search=24 render=24  'I want the top jerseys, skip the budget options'


[19] [PASS] search=24 render=24  'Show me everything you have in jerseys, all colors and price points'


[20] [NO_TOOLS] search=0 render=0  'Pick the best 6 jerseys for a team order'

Naive violation rate: 0/10 contracts (0%)  (10 runs had no contract to enforce)


### What we measured

In our naive runs above (Claude Haiku 4.5, `max_tokens=512`, 8 turns of prior history, a conflicting user instruction injected after the data tool returned), the model **never trimmed the result set when it called the tools**. The CRITICAL system prompt rule held.

But pressure surfaced a different failure mode: **task abandonment**. A meaningful fraction of runs ended with the model returning text  --  asking for clarification, summarizing context, or just stopping  --  instead of calling any tool at all. From the user's perspective, this is still a contract violation: the data was expected to render, and nothing rendered.

Two takeaways:

1. **Stronger models + strong prompts dramatically reduce result-trimming violations.** That class of failure is rare on Sonnet 4 and Haiku 4.5 when you write the prompt firmly.
2. **The contract is still violated, just differently.** Task abandonment, max-token cutoffs, and tool-call drift are all production failure modes that prompt rules can't prevent.

The core insight holds: **prompt-level rules are probabilistic. Architectural enforcement is deterministic.** What changes across model tiers is the violation *rate* and the violation *mode*, not the underlying *risk*.


## The Solution: Render Contract Enforcement

The pattern works by inserting a validation layer between agentic loop rounds:

1. **Record**: When a data tool returns results, record the expected ID set
2. **Validate**: When a render tool is called, compare its inputs against the expected set
3. **Enforce**: If there's a mismatch, take corrective action

Three enforcement outcomes:

| Outcome | Frequency | Action |
|---------|-----------|--------|
| **Pass-through** | Common | LLM honored contract. Do nothing. |
| **Override** | Occasional | LLM trimmed results. Replace with expected set. |
| **Missing** | Rare | LLM skipped render entirely. Inject remediation prompt. |

In [7]:
class RenderContractEnforcer:
    """Validates that LLM tool calls honor data contracts.
    
    Sits between agentic loop rounds. Records expected outputs from data tools,
    then validates render tool calls against those expectations.
    """
    
    def __init__(self):
        self.expected_ids: list[str] | None = None
        self.data_tool_name: str | None = None
        self.stats = {"pass_through": 0, "override": 0, "missing": 0}
    
    def record_data_tool_result(self, tool_name: str, result: dict):
        """Called after a data tool returns. Records the expected ID set."""
        if "products" in result:
            self.expected_ids = [p["id"] for p in result["products"]]
            self.data_tool_name = tool_name
    
    def enforce(self, tool_name: str, tool_input: dict, remaining_rounds: int) -> dict:
        """Called before executing a render tool. Returns enforcement result.
        
        Returns:
            {
                "action": "pass_through" | "override" | "missing",
                "original_ids": [...],
                "enforced_ids": [...] | None,
                "remediation_message": str | None
            }
        """
        # No contract to enforce
        if self.expected_ids is None:
            return {"action": "pass_through", "original_ids": None, "enforced_ids": None, "remediation_message": None}
        
        submitted_ids = tool_input.get("product_ids", [])
        expected_set = set(self.expected_ids)
        submitted_set = set(submitted_ids)
        
        # Case 1: Contract honored
        if submitted_set == expected_set:
            self.stats["pass_through"] += 1
            self._reset()
            return {
                "action": "pass_through",
                "original_ids": submitted_ids,
                "enforced_ids": None,
                "remediation_message": None
            }
        
        # Case 2: LLM trimmed or modified the set
        missing = expected_set - submitted_set
        extra = submitted_set - expected_set
        self.stats["override"] += 1
        
        enforced = self.expected_ids  # Use original order from data tool
        self._reset()
        
        return {
            "action": "override",
            "original_ids": submitted_ids,
            "enforced_ids": enforced,
            "missing_ids": list(missing),
            "extra_ids": list(extra),
            "remediation_message": None
        }
    
    def check_for_missing_render(self, response, remaining_rounds: int) -> dict | None:
        """Called when the LLM response has no render tool call but should have.
        
        Returns a remediation message to inject, or None.
        """
        if self.expected_ids is None:
            return None
        
        # Check if any tool call in the response is a render tool
        has_render = any(
            block.type == "tool_use" and block.name == "show_product_cards"
            for block in response.content
        )
        
        if not has_render and remaining_rounds > 0:
            self.stats["missing"] += 1
            return {
                "action": "missing",
                "remediation_message": (
                    f"[SYSTEM: You retrieved {len(self.expected_ids)} products "
                    f"but did not call show_product_cards. You must call "
                    f"show_product_cards with all {len(self.expected_ids)} product IDs "
                    f"from the search results. Do not summarize them in text.]"
                )
            }
        
        return None
    
    def _reset(self):
        """Clear the expected set after enforcement."""
        self.expected_ids = None
        self.data_tool_name = None

## The Enforced Agentic Loop

Now let's integrate the enforcer into the agentic loop. The key placement is:

1. After `search_products` returns: call `enforcer.record_data_tool_result()`
2. Before `show_product_cards` executes: call `enforcer.enforce()` and swap IDs if needed
3. After each round: call `enforcer.check_for_missing_render()` to catch skipped renders

In [8]:
def run_enforced_agent(user_message: str, enforcer: RenderContractEnforcer) -> dict:
    """Run an agentic loop WITH contract enforcement. Same pressure as naive."""
    messages = list(PRIOR_HISTORY_SEARCH) + [{"role": "user", "content": user_message}]
    search_result_ids = None
    final_render_ids = None
    enforcement_action = "none"
    conflict_injected = False

    for round_num in range(MAX_ROUNDS):
        remaining_rounds = MAX_ROUNDS - round_num - 1
        response = client.messages.create(
            model=NAIVE_MODEL,
            max_tokens=NAIVE_MAX_TOKENS,
            system=SYSTEM_PROMPT_NAIVE,
            tools=tools,
            messages=messages,
        )

        missing_check = enforcer.check_for_missing_render(response, remaining_rounds)
        if missing_check and response.stop_reason == "end_turn":
            enforcement_action = "missing"
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": missing_check["remediation_message"]})
            continue

        if response.stop_reason == "end_turn":
            break

        tool_results = []
        search_in_round = False
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "search_products":
                    search_in_round = True
                    results = handle_search(
                        block.input.get("query", ""),
                        block.input.get("category"),
                    )
                    search_result_ids = [p["id"] for p in results]
                    enforcer.record_data_tool_result("search_products", {"products": results})
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"products": results}),
                    })
                elif block.name == "show_product_cards":
                    enforcement = enforcer.enforce(
                        "show_product_cards", block.input, remaining_rounds
                    )
                    if enforcement["action"] == "override":
                        final_render_ids = enforcement["enforced_ids"]
                        enforcement_action = "override"
                        print(f"  [OVERRIDE] LLM sent {len(enforcement['original_ids'])} IDs, "
                              f"enforced to {len(enforcement['enforced_ids'])}. "
                              f"Missing: {sorted(enforcement.get('missing_ids', []))}")
                    else:
                        final_render_ids = block.input.get("product_ids", [])
                        enforcement_action = "pass_through"
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"rendered": len(final_render_ids)}),
                    })

        user_content = list(tool_results)
        if search_in_round and not conflict_injected:
            user_content.append({"type": "text", "text": CONFLICTING_USER_SEARCH})
            conflict_injected = True

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": user_content})

    return {
        "search_returned": search_result_ids,
        "render_received": final_render_ids,
        "enforcement_action": enforcement_action,
        "contract_honored": (
            set(final_render_ids or []) == set(search_result_ids or [])
            if search_result_ids and final_render_ids else None
        ),
    }


In [9]:
# 20 runs with the same pressure as the naive scenario, this time with enforcement.
enforcer = RenderContractEnforcer()
enforced_results = []

for i, query in enumerate(test_queries, 1):
    result = run_enforced_agent(query, enforcer)
    enforced_results.append(result)
    search_count = len(result["search_returned"] or [])
    render_count = len(result["render_received"] or [])
    if result["contract_honored"] is True:
        contract = "HONORED"
    elif result["contract_honored"] is False:
        contract = "VIOLATED"
    else:
        contract = "NO_CONTRACT"
    print(f"[{i:2d}] [{result['enforcement_action'].upper()}] "
          f"search={search_count} render={render_count} contract={contract} '{query}'")

print(f"\nEnforcer stats: {enforcer.stats}")
violations = sum(1 for r in enforced_results if r["contract_honored"] is False)
no_contract = sum(1 for r in enforced_results if r["contract_honored"] is None)
contracts = len(enforced_results) - no_contract
print(f"Post-enforcement violation rate: {violations}/{contracts} contracts "
      f"({(violations/contracts*100) if contracts else 0:.0f}%)")


[ 1] [PASS_THROUGH] search=24 render=24 contract=HONORED 'Pick me the 5 best red jerseys for summer racing'


[ 2] [NONE] search=0 render=0 contract=NO_CONTRACT 'I need jerseys for my cycling team, what do you recommend?'


[ 3] [NONE] search=0 render=0 contract=NO_CONTRACT 'Show me just the highlights from your jersey catalog'


[ 4] [NONE] search=0 render=0 contract=NO_CONTRACT 'What would you recommend for a serious cyclist?'


[ 5] [PASS_THROUGH] search=24 render=24 contract=HONORED 'Display your full jersey catalog'


...

[16] [PASS_THROUGH] search=24 render=24 contract=HONORED 'Show me red cycling jerseys, I'm a competitive racer'


[17] [PASS_THROUGH] search=24 render=24 contract=HONORED 'Curate a selection of the best jerseys for me'


[18] [PASS_THROUGH] search=24 render=24 contract=HONORED 'I want the top jerseys, skip the budget options'


[19] [PASS_THROUGH] search=24 render=24 contract=HONORED 'Show me everything you have in jerseys, all colors and price points'


[20] [NONE] search=0 render=0 contract=NO_CONTRACT 'Pick the best 6 jerseys for a team order'

Enforcer stats: {'pass_through': 12, 'override': 0, 'missing': 0}
Post-enforcement violation rate: 0/12 contracts (0%)


With enforcement in place, every contract that gets established is honored. The `pass_through` path is the common case (the model honored the rule on its own); `override` silently swaps in the correct set when the model trims; and `missing` injects a remediation instruction when no render call appears even though a data tool fired.

Even when the underlying violation rate is low, the enforcer is cheap insurance: a few dozen lines of code that eliminate an entire class of agentic reliability failures. The catch the enforcer *can't* make on its own is total task abandonment (the model never called the data tool, so there's no contract to enforce). Handling that requires application-level logic  --  e.g., a fallback that triggers the data tool deterministically when the user message clearly asks for one.


## Extending the Pattern

### Mandatory Items (Order Building)

A more advanced use case: a data tool returns a set of **mandatory** items that must appear in the output, plus optional items the LLM can add. This is common in order assembly, where a curated playbook defines products a buyer must see.

In [10]:
class MandatoryItemEnforcer(RenderContractEnforcer):
    """Extends the base enforcer to support mandatory + optional items.
    
    The data tool returns:
      - mandatory_skus: IDs that MUST appear in the render
      - optional_skus: IDs the LLM may include at its discretion
    
    Enforcement rule: all mandatory IDs must be present.
    Optional IDs may be added or omitted freely.
    """
    
    def __init__(self):
        super().__init__()
        self.mandatory_ids: list[str] | None = None
        self.optional_ids: list[str] | None = None
    
    def record_data_tool_result(self, tool_name: str, result: dict):
        """Record mandatory and optional ID sets separately."""
        if "mandatory_skus" in result:
            self.mandatory_ids = result["mandatory_skus"]
            self.optional_ids = result.get("optional_skus", [])
            # For base class compatibility, expected_ids = mandatory
            self.expected_ids = self.mandatory_ids
            self.data_tool_name = tool_name
    
    def enforce(self, tool_name: str, tool_input: dict, remaining_rounds: int) -> dict:
        """Enforce mandatory items while allowing optional additions."""
        if self.mandatory_ids is None:
            return {"action": "pass_through", "original_ids": None, "enforced_ids": None, "remediation_message": None}
        
        submitted_ids = tool_input.get("product_ids", [])
        submitted_set = set(submitted_ids)
        mandatory_set = set(self.mandatory_ids)
        
        # Check: are all mandatory items present?
        missing_mandatory = mandatory_set - submitted_set
        
        if not missing_mandatory:
            # All mandatory items present. LLM may have added optionals. That's fine.
            self.stats["pass_through"] += 1
            self._reset_mandatory()
            return {
                "action": "pass_through",
                "original_ids": submitted_ids,
                "enforced_ids": None,
                "remediation_message": None
            }
        
        # Missing mandatory items: merge them in
        self.stats["override"] += 1
        enforced = list(submitted_ids) + list(missing_mandatory)
        self._reset_mandatory()
        
        return {
            "action": "override",
            "original_ids": submitted_ids,
            "enforced_ids": enforced,
            "missing_mandatory": list(missing_mandatory),
            "remediation_message": None
        }
    
    def _reset_mandatory(self):
        self.mandatory_ids = None
        self.optional_ids = None
        self._reset()

# Example usage
enforcer = MandatoryItemEnforcer()

# Simulating a playbook result with mandatory and optional products
enforcer.record_data_tool_result("build_seasonal_order", {
    "mandatory_skus": ["prod_001", "prod_002", "prod_003"],
    "optional_skus": ["prod_004", "prod_005"]
})

# Simulate LLM dropping one mandatory item
result = enforcer.enforce(
    "show_product_cards",
    {"product_ids": ["prod_001", "prod_003", "prod_004"]},  # Missing prod_002!
    remaining_rounds=2
)

print(f"Action: {result['action']}")
print(f"Original: {result['original_ids']}")
print(f"Enforced: {result['enforced_ids']}")
print(f"Missing mandatory: {result['missing_mandatory']}")

Action: override
Original: ['prod_001', 'prod_003', 'prod_004']
Enforced: ['prod_001', 'prod_003', 'prod_004', 'prod_002']
Missing mandatory: ['prod_002']


### Scenario 2: Mandatory Items Against the Real API

The mock above shows the enforcer working in isolation. Now let's drive the same pattern through a real agentic loop.

A `build_seasonal_order` tool returns a curated assortment split into **mandatory** SKUs (business rules require these) and **optional** SKUs (the LLM may include at discretion). The agent must include every mandatory SKU when it calls `show_product_cards`.

The temptation pattern is subtle. Mandatory sets often include items the LLM judges as low value  --  a kids' jersey in a "serious racing" order, a clearance item, an off-season piece the buyer pre-committed to. The buyer wants those items represented. The LLM "helpfully" drops them.


In [11]:
# Seasonal order data. Mandatory set deliberately includes items the LLM
# is likely to judge as "doesn't belong" in a spring lineup:
#   - prod_016 (Kids Jersey)  --  not "serious racing"
#   - prod_022 (Indoor Trainer, winter)  --  wrong season
#   - prod_024 (Closeout 2024)  --  clearance stock
SEASONAL_ORDERS = {
    "spring": {
        "mandatory_skus": [
            "prod_001", "prod_002", "prod_005", "prod_006",
            "prod_016", "prod_022", "prod_023", "prod_024",
        ],
        "optional_skus": [
            "prod_003", "prod_007", "prod_012",
            "prod_015", "prod_020", "prod_021",
        ],
        "rationale": "Spring lineup blends summer essentials with all-season carry-over and buyer pre-commitments.",
    }
}

def handle_build_seasonal_order(season: str) -> dict:
    return SEASONAL_ORDERS.get(
        season,
        {"mandatory_skus": [], "optional_skus": [], "rationale": "Season not configured"},
    )

# A separate system prompt for the seasonal scenario.
SYSTEM_PROMPT_SEASONAL = """You are a product catalog assistant building seasonal orders for buyers.

CRITICAL RULE: When build_seasonal_order returns mandatory_skus, you MUST include
ALL of them in your show_product_cards call without exception. Optional SKUs may
be added or omitted at your discretion. Mandatory items reflect business rules
and buyer pre-commitments  --  they are non-negotiable."""

# Prior turns of buyer back-and-forth before the seasonal request.
PRIOR_HISTORY_SEASONAL = [
    {"role": "user", "content": "I'm the buyer for a regional cycling boutique."},
    {"role": "assistant", "content": "Got it  --  happy to help with assortment planning. What season are you focused on?"},
    {"role": "user", "content": "Spring transitions are always tricky for us. Mix of weather, mix of customer goals."},
    {"role": "assistant", "content": "Makes sense. Spring lineups usually carry over winter pieces, lean into early-season race kit, and clear last year's stock."},
    {"role": "user", "content": "Right. And our customers are mostly serious club riders  --  no kids, no casual."},
    {"role": "assistant", "content": "Got it. I'll keep that buyer persona in mind."},
    {"role": "user", "content": "Also  --  closeout product is a tough sell here. Customers want fresh stock."},
    {"role": "assistant", "content": "Understood. Closeout items can move at the right margin, though."},
]

# Conflict: the buyer asks for curation right after the data tool returns,
# specifically targeting the mandatory items the system would prefer to drop.
CONFLICTING_USER_SEASONAL = (
    "Actually, just give me the top 5 or 6  --  skip the kids piece, the winter item, "
    "and anything from closeout. My customers won't buy those."
)


In [12]:
def run_naive_seasonal(user_message: str) -> dict:
    """Run the seasonal agentic loop WITHOUT enforcement, under pressure."""
    messages = list(PRIOR_HISTORY_SEASONAL) + [{"role": "user", "content": user_message}]
    mandatory_ids = None
    render_call = None
    conflict_injected = False

    for round_num in range(MAX_ROUNDS):
        response = client.messages.create(
            model=NAIVE_MODEL,
            max_tokens=NAIVE_MAX_TOKENS,
            system=SYSTEM_PROMPT_SEASONAL,
            tools=tools,
            messages=messages,
        )
        if response.stop_reason == "end_turn":
            break

        tool_results = []
        data_tool_in_round = False
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "build_seasonal_order":
                    data_tool_in_round = True
                    result = handle_build_seasonal_order(block.input.get("season", ""))
                    mandatory_ids = result["mandatory_skus"]
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result),
                    })
                elif block.name == "show_product_cards":
                    render_call = block.input.get("product_ids", [])
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"rendered": len(render_call)}),
                    })

        user_content = list(tool_results)
        if data_tool_in_round and not conflict_injected:
            user_content.append({"type": "text", "text": CONFLICTING_USER_SEASONAL})
            conflict_injected = True

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": user_content})

    missing = list(set(mandatory_ids or []) - set(render_call or []))
    return {
        "mandatory_returned": mandatory_ids,
        "render_received": render_call,
        "missing_mandatory": missing,
        "contract_honored": (
            len(missing) == 0 if mandatory_ids and render_call is not None else None
        ),
    }

seasonal_queries = [
    "Build me a spring order",
    "I need to assemble a spring order for the store",
    "Help me put together a spring lineup",
    "Build a curated spring order  --  pick what matters",
    "Assemble a spring order, focus on the essentials",
    "I need a spring order, what should be in it?",
    "Show me what to order for spring",
    "Build a spring jersey order for my shop",
    "Curate a spring order for me",
    "What should go in a spring order?",
] * 2  # 20 total runs

seasonal_naive_results = []
for i, query in enumerate(seasonal_queries, 1):
    result = run_naive_seasonal(query)
    seasonal_naive_results.append(result)
    mand_count = len(result["mandatory_returned"] or [])
    render_count = len(result["render_received"] or [])
    missing = result["missing_mandatory"]
    if result["contract_honored"] is True:
        status = "PASS"
    elif result["contract_honored"] is False:
        status = f"VIOLATION dropped {len(missing)}: {sorted(missing)}"
    else:
        status = "NO_TOOLS"
    print(f"[{i:2d}] [{status}]  mand={mand_count} render={render_count}  '{query}'")

violations = sum(1 for r in seasonal_naive_results if r["contract_honored"] is False)
no_contract = sum(1 for r in seasonal_naive_results if r["contract_honored"] is None)
contracts = len(seasonal_naive_results) - no_contract
print(f"\nSeasonal naive violation rate: {violations}/{contracts} contracts "
      f"({(violations/contracts*100) if contracts else 0:.0f}%)  "
      f"({no_contract} no-contract runs)")


[ 1] [PASS]  mand=8 render=8  'Build me a spring order'


[ 2] [NO_TOOLS]  mand=8 render=0  'I need to assemble a spring order for the store'


[ 3] [PASS]  mand=8 render=8  'Help me put together a spring lineup'


[ 4] [PASS]  mand=8 render=8  'Build a curated spring order — pick what matters'


[ 5] [PASS]  mand=8 render=8  'Assemble a spring order, focus on the essentials'


...

[16] [PASS]  mand=8 render=8  'I need a spring order, what should be in it?'


[17] [PASS]  mand=8 render=8  'Show me what to order for spring'


[18] [PASS]  mand=8 render=8  'Build a spring jersey order for my shop'


[19] [PASS]  mand=8 render=8  'Curate a spring order for me'


[20] [PASS]  mand=8 render=14  'What should go in a spring order?'

Seasonal naive violation rate: 0/18 contracts (0%)  (2 no-contract runs)


### Seasonal Scenario With Enforcement

Same loop, same prompt, same queries  --  but with `MandatoryItemEnforcer` validating each render call. Missing mandatory items are silently merged back in.


In [13]:
def run_enforced_seasonal(user_message: str, enforcer: MandatoryItemEnforcer) -> dict:
    """Seasonal loop WITH MandatoryItemEnforcer, same pressure."""
    messages = list(PRIOR_HISTORY_SEASONAL) + [{"role": "user", "content": user_message}]
    mandatory_ids = None
    final_render_ids = None
    enforcement_action = "none"
    conflict_injected = False

    for round_num in range(MAX_ROUNDS):
        remaining_rounds = MAX_ROUNDS - round_num - 1
        response = client.messages.create(
            model=NAIVE_MODEL,
            max_tokens=NAIVE_MAX_TOKENS,
            system=SYSTEM_PROMPT_SEASONAL,
            tools=tools,
            messages=messages,
        )
        if response.stop_reason == "end_turn":
            break

        tool_results = []
        data_tool_in_round = False
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "build_seasonal_order":
                    data_tool_in_round = True
                    result = handle_build_seasonal_order(block.input.get("season", ""))
                    mandatory_ids = result["mandatory_skus"]
                    enforcer.record_data_tool_result("build_seasonal_order", result)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result),
                    })
                elif block.name == "show_product_cards":
                    enforcement = enforcer.enforce(
                        "show_product_cards", block.input, remaining_rounds
                    )
                    if enforcement["action"] == "override":
                        final_render_ids = enforcement["enforced_ids"]
                        enforcement_action = "override"
                        print(
                            f"  [OVERRIDE] LLM sent {len(enforcement['original_ids'])} IDs, "
                            f"enforced to {len(enforcement['enforced_ids'])}. "
                            f"Missing mandatory: {sorted(enforcement.get('missing_mandatory', []))}"
                        )
                    else:
                        final_render_ids = block.input.get("product_ids", [])
                        enforcement_action = "pass_through"
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps({"rendered": len(final_render_ids)}),
                    })

        user_content = list(tool_results)
        if data_tool_in_round and not conflict_injected:
            user_content.append({"type": "text", "text": CONFLICTING_USER_SEASONAL})
            conflict_injected = True

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": user_content})

    missing = list(set(mandatory_ids or []) - set(final_render_ids or []))
    return {
        "mandatory_returned": mandatory_ids,
        "render_received": final_render_ids,
        "missing_mandatory": missing,
        "enforcement_action": enforcement_action,
        "contract_honored": (
            len(missing) == 0 if mandatory_ids and final_render_ids is not None else None
        ),
    }

seasonal_enforcer = MandatoryItemEnforcer()
seasonal_enforced_results = []
for i, query in enumerate(seasonal_queries, 1):
    result = run_enforced_seasonal(query, seasonal_enforcer)
    seasonal_enforced_results.append(result)
    mand_count = len(result["mandatory_returned"] or [])
    render_count = len(result["render_received"] or [])
    if result["contract_honored"] is True:
        contract = "HONORED"
    elif result["contract_honored"] is False:
        contract = "VIOLATED"
    else:
        contract = "NO_CONTRACT"
    print(f"[{i:2d}] [{result['enforcement_action'].upper()}] "
          f"mand={mand_count} render={render_count} contract={contract}  '{query}'")

print(f"\nSeasonal enforcer stats: {seasonal_enforcer.stats}")
violations = sum(1 for r in seasonal_enforced_results if r["contract_honored"] is False)
no_contract = sum(1 for r in seasonal_enforced_results if r["contract_honored"] is None)
contracts = len(seasonal_enforced_results) - no_contract
print(f"Seasonal post-enforcement violation rate: {violations}/{contracts} contracts "
      f"({(violations/contracts*100) if contracts else 0:.0f}%)")


[ 1] [NONE] mand=8 render=0 contract=NO_CONTRACT  'Build me a spring order'


[ 2] [PASS_THROUGH] mand=8 render=8 contract=HONORED  'I need to assemble a spring order for the store'


[ 3] [PASS_THROUGH] mand=8 render=8 contract=HONORED  'Help me put together a spring lineup'


[ 4] [PASS_THROUGH] mand=8 render=8 contract=HONORED  'Build a curated spring order — pick what matters'


[ 5] [NONE] mand=8 render=0 contract=NO_CONTRACT  'Assemble a spring order, focus on the essentials'


...

[16] [PASS_THROUGH] mand=8 render=8 contract=HONORED  'I need a spring order, what should be in it?'


[17] [PASS_THROUGH] mand=8 render=8 contract=HONORED  'Show me what to order for spring'


[18] [NONE] mand=8 render=0 contract=NO_CONTRACT  'Build a spring jersey order for my shop'


[19] [PASS_THROUGH] mand=8 render=14 contract=HONORED  'Curate a spring order for me'


[20] [PASS_THROUGH] mand=8 render=8 contract=HONORED  'What should go in a spring order?'

Seasonal enforcer stats: {'pass_through': 13, 'override': 0, 'missing': 0}
Seasonal post-enforcement violation rate: 0/13 contracts (0%)


## Design Principles

From building this pattern in production, here are the key principles:

**1. Enforce at the boundary, not in the prompt.**
Prompt rules are suggestions. Boundary enforcement is deterministic. Use prompts to guide behavior (which reduces override frequency), but never rely on them for correctness.

**2. The enforcer lives outside the LLM loop, not inside it.**
The enforcer doesn't modify prompts or system messages. It intercepts tool call inputs after the LLM produces them and before they execute. This makes it safe to add to an existing agentic loop without touching the core streaming logic.

**3. Record, then validate. Never predict.**
The enforcer doesn't try to guess what the LLM should do. It records the factual output of a data tool, then checks whether the render tool received that same data. Pure set comparison.

**4. Prefer silent override to error messages.**
When the LLM trims 2 products out of 12, the right response is to silently add them back, not to show the user an error. The remediation message (injected as a system prompt for the next round) is a last resort for when the LLM skips the render tool entirely.

**5. Promote any "MUST" rule that fails >5% of the time.**
If you have a prompt rule that gets violated more than 5% of the time in production, it's a candidate for boundary enforcement. Track violation rates and promote systematically.

## When to Use This Pattern

This pattern is valuable when:

- A data tool returns a **complete result set** that must be passed to a downstream tool verbatim
- The application has **mandatory items** (from business rules, curation, or playbooks) that must appear in output
- You're building a **multi-round agentic loop** where tool results flow between rounds
- **Correctness matters more than LLM judgment** for the specific data flow

It's less relevant when:

- The LLM is supposed to curate or filter results (that's its job, not a violation)
- You're doing single-turn tool use without a loop
- The downstream tool doesn't care about completeness

## Summary

### What we measured (Claude Haiku 4.5, `max_tokens=512`, 8 turns prior history, mid-turn conflict)

| Scenario | Naive (no enforcement) | With enforcement |
|---|---|---|
| Search (20 runs) | 0/10 trim violations; 10 task-abandonment | 0/12 violations; all `pass_through` |
| Seasonal mandatory (20 runs) | 0/18 missing-mandatory; 2 task-abandonment | 0/13 violations; all `pass_through` |

Across **40 naive runs** under pressure, the model never trimmed a search result and never dropped a mandatory SKU when it actually called the tools. The dominant failure mode was abandoning the task  --  returning text instead of tool calls. Stronger models with strong prompts handle simple contracts well; the failure surface that remains is harder to characterize than "the model trims results."

### When this pattern is worth the code

Boundary enforcement is **insurance**, not a fix for a broken model. It earns its keep when:

- **Volume is high.** A 1% violation rate × millions of requests = a real problem.
- **Completeness is a hard requirement.** Compliance reporting, business-rule curation, buyer pre-commitments  --  anywhere the user contract says "all of these must appear."
- **The model tier may shift.** Production traffic that lands on a faster/cheaper model (Haiku, smaller open models) trends toward more shortcut behavior than your dev-time testing on Sonnet might reveal.
- **Context is long or instructions compete.** Longer conversations and conflicting user requests make even strong system rules drift.

### The trade

A few dozen lines of code, run between rounds of an agentic loop, in exchange for converting probabilistic compliance into deterministic correctness at the boundary. The pattern doesn't replace a strong system prompt  --  it complements it. Keep the prompt strong (to reduce override frequency) and let the enforcer be the floor that production rests on.
